# Allocation validation

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_DIR = Path.cwd() / 'scratchpad/hyperliquid-ic' if (Path.cwd() / 'scratchpad/hyperliquid-ic').exists() else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
ARTIFACTS = PROJECT_DIR / '_artifacts-rewrite'
ARTIFACTS.mkdir(exist_ok=True)
from ic_research import ResearchConfig, completed_daily_window, simulate_allocator, summarise_backtest

CONFIG = ResearchConfig()
features = pd.read_parquet(ARTIFACTS / 'features.parquet')
observations = pd.read_parquet(ARTIFACTS / 'observations.parquet')
evaluation_start, evaluation_end = completed_daily_window(observations)
features = features[features.date.between(evaluation_start, evaluation_end)].copy()
growth = pd.read_parquet(ARTIFACTS / 'predictions-short-core.parquet')
growth = growth[(growth.target == 'forward_log_growth_30') & (growth.status == 'predicted')][['date', 'address', 'prediction']]
print('Allocation evaluation dates:', features.date.min(), 'to', features.date.max(), '| growth predictions:', len(growth))


Allocation evaluation dates: 2025-09-13 00:00:00 to 2026-09-12 00:00:00 | growth predictions: 87081


In [2]:
arms = {
    'incumbent': (pd.DataFrame(columns=['date', 'address', 'prediction']), 'incumbent', 6),
    'candidate_fixed': (growth, 'candidate', 10),
    'candidate_adaptive': (growth, 'adaptive', 20),
}
curves, summaries = {}, []
for name, (scores, mode, limit) in arms.items():
    curve, trades = simulate_allocator(features, scores, observations=observations, config=CONFIG, mode=mode, max_positions=limit)
    curve.to_parquet(ARTIFACTS / f'equity-{name}.parquet', index=False)
    trades.to_parquet(ARTIFACTS / f'trades-{name}.parquet', index=False)
    summary = summarise_backtest(curve)
    summary['arm'] = name
    summaries.append(summary)
    curves[name] = curve
metrics = pd.DataFrame(summaries).set_index('arm')
metrics.to_csv(ARTIFACTS / 'allocation-metrics.csv')
display(metrics)


,start,end,final_equity,cagr,volatility,sharpe,max_drawdown,mean_positions,mean_turnover,mean_cash_fraction,mean_stale_positions,mean_carried_positions
arm,,,,,,,,,,,,
incumbent,2025-09-13 00:00:00,2026-09-12 00:00:00,157402.271889,0.049487,0.200354,0.340885,-0.156379,6.000000,26806.186821,0.024776,0.000000,1.972603
candidate_fixed,2025-09-13 00:00:00,2026-09-12 00:00:00,148774.272856,-0.008194,0.203200,0.059469,-0.223487,14.449315,30882.034718,0.138998,4.446575,3.287671
candidate_adaptive,2025-09-13 00:00:00,2026-09-12 00:00:00,141835.222550,-0.054577,0.140034,-0.330820,-0.173210,26.509589,26460.252403,0.043519,7.580822,6.167123


In [3]:
comparison = pd.concat({name: curve.set_index('date').equity for name, curve in curves.items()}, axis=1)
comparison.to_parquet(ARTIFACTS / 'allocation-equity-comparison.parquet')
display(comparison.tail())
print('Interpretation guard: candidate selection is driven by out-of-fold 30-day growth predictions; the replay marks the book from causally forward-filled daily NAVs while retaining raw freshness as a diagnostic; existing holdings can remain marked after a vault falls below candidate TVL eligibility and must be read with the coverage artefacts.')


,incumbent,candidate_fixed,candidate_adaptive
date,,,
2026-09-08,164581.879128,149606.302964,142213.414533
2026-09-09,162390.754597,149564.053806,142237.785877
2026-09-10,157033.293383,149549.766731,141975.182692
2026-09-11,157002.685756,149599.577215,142066.718705
2026-09-12,157402.271889,148774.272856,141835.222550


Interpretation guard: candidate selection is driven by out-of-fold 30-day growth predictions; the replay marks the book from causally forward-filled daily NAVs while retaining raw freshness as a diagnostic; existing holdings can remain marked after a vault falls below candidate TVL eligibility and must be read with the coverage artefacts.
